# GCG Attack CLI Examples

This notebook demonstrates how to run Greedy Coordinate Gradient (GCG) attacks using the AdvSecureNet CLI interface.

## Overview

The GCG attack is a method for generating adversarial suffixes that can bypass safety filters in large language models. This notebook shows how to:

- Run quick sanity checks
- Execute full GCG attacks with custom configurations
- Experiment with different models and parameters

## Prerequisites

- Python environment with AdvSecureNet installed
- Access to HuggingFace models (or local model files)
- Sufficient computational resources (GPU recommended)

---

## Quick Start

### 1. Sanity Check
First, let's run a quick test to ensure everything is working:

In [ ]:
import os
from pathlib import Path

# Find the project root directory (contains setup.py or pyproject.toml)
current_dir = Path.cwd()
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / 'setup.py').exists() or (project_root / 'pyproject.toml').exists():
        break
    project_root = project_root.parent

# Change to GCG experiments directory
gcg_experiments_dir = project_root / 'advsecurenet' / 'llm' / 'GCG' / 'experiments'
os.chdir(gcg_experiments_dir)
print(f"Working from: {gcg_experiments_dir}")

!python3 -m advsecurenet.llm.GCG.experiments.cli_gcg quick

#### And finally running the official config

In [ ]:
!python3 -m advsecurenet.llm.GCG.experiments.cli_gcg attack --config configs/universal_config.py --model Qwen/Qwen2-0.5B --steps 15

In [ ]:
import os
import shutil
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / 'setup.py').exists() or (project_root / 'pyproject.toml').exists():
        break
    project_root = project_root.parent

gcg_experiments_dir = project_root / 'advsecurenet' / 'llm' / 'GCG' / 'experiments'
os.chdir(gcg_experiments_dir)
print(f"Working from: {gcg_experiments_dir}")

def modify_config_for_model(model_name, config_path="configs/universal_config.py"):
    backup_path = f"{config_path}.backup"
    shutil.copy2(config_path, backup_path)
    
    with open(config_path, 'r') as f:
        content = f.read()
    
    modified_content = content.replace(
        'config.model_name = "Qwen/Qwen2.5-0.5B-Instruct"',
        f'config.model_name = "{model_name}"'
    )
    
    with open(config_path, 'w') as f:
        f.write(modified_content)
    
    print(f"Configuration modified for model: {model_name}")
    return backup_path

def restore_config(backup_path, config_path="configs/universal_config.py"):
    shutil.copy2(backup_path, config_path)
    os.remove(backup_path)
    print("Original configuration restored")

print("Running GCG Attack Tests with Model Override")
print("=" * 60)

models_to_test = ["gpt2", "distilgpt2", "microsoft/DialoGPT-small"]

for model in models_to_test:
    print(f"\nTesting model: {model}")
    print("-" * 40)
    
    backup_path = None
    try:
        backup_path = modify_config_for_model(model)
        
        !python3 -m advsecurenet.llm.GCG.experiments.cli_gcg attack \
            --config configs/universal_config.py \
            --steps 5 \
            --train-data 1 \
            --batch-size 2 \
            --verbose
        
        print(f"Status: {model} - SUCCESS")
        
    except Exception as e:
        print(f"Status: {model} - FAILED ({str(e)})")
        
    finally:
        if backup_path:
            restore_config(backup_path)

print("\nTesting completed. All configurations restored.")

In [ ]:
# Add this cell to explore different attack parameters
import os
from pathlib import Path

# Setup working directory
current_dir = Path.cwd()
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / 'setup.py').exists() or (project_root / 'pyproject.toml').exists():
        break
    project_root = project_root.parent

gcg_experiments_dir = project_root / 'advsecurenet' / 'llm' / 'GCG' / 'experiments'
os.chdir(gcg_experiments_dir)
print(f"Working from: {gcg_experiments_dir}")

# Try different attack configurations
attack_configs = [
    {"model": "microsoft/DialoGPT-small", "steps": 1, "name": "Medium Test"},
    {"model": "Qwen/Qwen2-0.5B", "steps": 1, "name": "Quick Test"},
    {"model": "gpt2", "steps": 1, "name": "GPT-2 Test"}
]

for config in attack_configs:
    print(f"\n=== Running {config['name']} ===")
    print(f"Model: {config['model']}, Steps: {config['steps']}")
    
    !python3 -m advsecurenet.llm.GCG.experiments.cli_gcg attack \
        --model {config['model']} \
        --config configs/universal_config.py \
        --steps {config['steps']} \
        --config-override n_test_data=3

In [ ]:
os.chdir(gcg_experiments_dir/"launch_scripts")
!chmod +x run_universal_gcg.sh

In [ ]:
!./run_universal_gcg.sh